# Data Transformation Pipeline

This notebook performs data transformation and cleansing operations on the Bronze layer data.

## Execution Steps:
1. Load data from Bronze layer
2. Apply business rules and transformations
3. Perform data enrichment
4. Store transformed data in Silver layer

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import json

# MS Fabric specific imports
# from fabric import FabricDataFrame
# from fabric.lakehouse import Lakehouse

print(f"Starting data transformation at {datetime.now()}")

In [ ]:
# Load data from Bronze layer
def load_from_bronze(table_name):
    """
    Load data from Bronze layer
    
    Args:
        table_name (str): Bronze table name
    
    Returns:
        pd.DataFrame: Loaded data
    """
    bronze_path = f"/lakehouse/default/Tables/bronze_{table_name}"
    
    print(f"Loading data from Bronze layer: {bronze_path}")
    
    # In real implementation, would use:
    # df = pd.read_delta(bronze_path)
    
    # Simulate loading Bronze data
    sample_data = {
        'id': range(1, 1001),
        'customer_id': [f'CUST_{i:04d}' for i in range(1, 1001)],
        'transaction_amount': [100.0 + (i * 0.5) for i in range(1, 1001)],
        'transaction_date': [datetime.now() - timedelta(days=np.random.randint(0, 30)) for _ in range(1000)],
        'source_system': ['azure_sql'] * 1000,
        '_ingestion_timestamp': [datetime.now() for _ in range(1000)],
        '_source_file': ['customer_transactions_batch'] * 1000
    }
    
    return pd.DataFrame(sample_data)

# Load Bronze data
bronze_transactions = load_from_bronze('customer_transactions')
print(f"Loaded {len(bronze_transactions)} records from Bronze layer")
print(bronze_transactions.head())

In [ ]:
# Data transformation functions
def apply_business_rules(df):
    """
    Apply business rules and data transformations
    
    Args:
        df (pd.DataFrame): Input DataFrame
    
    Returns:
        pd.DataFrame: Transformed DataFrame
    """
    # Create a copy to avoid modifying original data
    transformed_df = df.copy()
    
    # Business rule: Categorize transaction amounts
    def categorize_amount(amount):
        if amount < 100:
            return 'Small'
        elif amount < 500:
            return 'Medium'
        elif amount < 1000:
            return 'Large'
        else:
            return 'Very Large'
    
    transformed_df['transaction_category'] = transformed_df['transaction_amount'].apply(categorize_amount)
    
    # Business rule: Calculate transaction month and year
    transformed_df['transaction_year'] = transformed_df['transaction_date'].dt.year
    transformed_df['transaction_month'] = transformed_df['transaction_date'].dt.month
    transformed_df['transaction_day_of_week'] = transformed_df['transaction_date'].dt.day_name()
    
    # Business rule: Flag high-value transactions
    transformed_df['is_high_value'] = transformed_df['transaction_amount'] > 750
    
    # Add transformation metadata
    transformed_df['_transformation_timestamp'] = datetime.now()
    transformed_df['_transformation_version'] = '1.0'
    
    print(f"Applied business rules to {len(transformed_df)} records")
    
    return transformed_df

# Apply transformations
silver_transactions = apply_business_rules(bronze_transactions)
print(f"Transaction categories distribution:")
print(silver_transactions['transaction_category'].value_counts())
print(f"\nHigh-value transactions: {silver_transactions['is_high_value'].sum()}")

In [ ]:
# Data enrichment
def enrich_customer_data(df):
    """
    Enrich transaction data with customer information
    
    Args:
        df (pd.DataFrame): Transaction DataFrame
    
    Returns:
        pd.DataFrame: Enriched DataFrame
    """
    # Simulate customer master data
    customer_data = {
        'customer_id': [f'CUST_{i:04d}' for i in range(1, 1001)],
        'customer_segment': np.random.choice(['Premium', 'Standard', 'Basic'], 1000),
        'customer_region': np.random.choice(['North', 'South', 'East', 'West'], 1000),
        'customer_age_group': np.random.choice(['18-25', '26-35', '36-45', '46-55', '55+'], 1000)
    }
    
    customer_df = pd.DataFrame(customer_data)
    
    # Join with customer data
    enriched_df = df.merge(customer_df, on='customer_id', how='left')
    
    # Calculate customer-level aggregations
    customer_stats = df.groupby('customer_id').agg({
        'transaction_amount': ['count', 'sum', 'mean'],
        'transaction_date': ['min', 'max']
    }).round(2)
    
    # Flatten column names
    customer_stats.columns = ['_'.join(col).strip() for col in customer_stats.columns.values]
    customer_stats = customer_stats.reset_index()
    
    # Rename columns for clarity
    customer_stats.columns = [
        'customer_id', 'total_transactions', 'total_amount', 'avg_amount',
        'first_transaction_date', 'last_transaction_date'
    ]
    
    # Join customer statistics
    enriched_df = enriched_df.merge(customer_stats, on='customer_id', how='left')
    
    print(f"Enriched {len(enriched_df)} records with customer data")
    
    return enriched_df

# Enrich data
enriched_transactions = enrich_customer_data(silver_transactions)
print(f"\nCustomer segment distribution:")
print(enriched_transactions['customer_segment'].value_counts())
print(f"\nSample enriched data:")
print(enriched_transactions[['customer_id', 'transaction_amount', 'customer_segment', 'customer_region']].head())

In [ ]:
# Save to Silver layer
def save_to_silver(df, table_name, partition_columns=None):
    """
    Save DataFrame to Silver layer in Delta format
    
    Args:
        df (pd.DataFrame): Data to save
        table_name (str): Target table name
        partition_columns (list): Columns to partition by
    """
    silver_path = f"/lakehouse/default/Tables/silver_{table_name}"
    
    print(f"Saving {len(df)} records to Silver layer: {silver_path}")
    
    # In real implementation, would use:
    # df.to_delta(silver_path, mode='overwrite', partition_cols=partition_columns)
    
    return silver_path

# Save to Silver layer
silver_path = save_to_silver(
    enriched_transactions, 
    'customer_transactions',
    partition_columns=['transaction_year', 'transaction_month']
)

print(f"Data saved to Silver layer: {silver_path}")
print("\nData transformation completed successfully!")
print(f"Next step: Run notebook 03_data_aggregation.ipynb")